In [4]:
import os
import sys
from pathlib import Path

# Ensure project root is importable when running from notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from nepse_analyst.config import BASE_DIR, NEWS_CACHE_DIR, VECTOR_STORE_DIR, SCRAPE_DELAY_SEC, MAX_ARTICLES

os.makedirs(NEWS_CACHE_DIR, exist_ok=True)
os.makedirs(VECTOR_STORE_DIR, exist_ok=True)

In [5]:
import requests
from bs4 import BeautifulSoup
import time
import json
import hashlib
from datetime import datetime
from urllib.parse import urljoin, urlparse
from nepse_analyst.language_detector import detect_language

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}


def _to_abs_url(base_url: str, href: str) -> str | None:
    if not href:
        return None
    href = href.strip()
    if href.startswith("#") or href.startswith("javascript:") or href.startswith("mailto:"):
        return None
    return urljoin(base_url, href)


def _extract_news_links(
    soup: BeautifulSoup,
    base_url: str,
    allowed_domains: tuple[str, ...],
    selectors: tuple[str, ...],
    path_keywords: tuple[str, ...],
) -> list[str]:
    links = set()

    candidate_anchors = []
    for sel in selectors:
        candidate_anchors.extend(soup.select(sel))
    if not candidate_anchors:
        candidate_anchors = soup.select("a[href]")

    for a in candidate_anchors:
        href = a.get("href", "")
        abs_url = _to_abs_url(base_url, href)
        if not abs_url:
            continue
        parsed = urlparse(abs_url)
        if not any(domain in parsed.netloc for domain in allowed_domains):
            continue

        path_lower = parsed.path.lower()
        if not any(k in path_lower for k in path_keywords):
            continue

        # Keep detail pages and drop obvious listing/category pages.
        if "category" in path_lower or "page/" in path_lower:
            continue
        if path_lower.rstrip("/") in ("", "/news", "/latest", "/announcement", "/news-page"):
            continue

        # Sharesansar detail pattern guard to avoid menu/link noise.
        if "sharesansar.com" in parsed.netloc and "/newsdetail/" not in path_lower:
            continue

        links.add(abs_url)

    return sorted(links)


def get_article_urls_sharesansar(max_pages: int = 50) -> list[str]:
    """Scrape listing pages to collect article URLs from Sharesansar."""
    urls = set()
    empty_streak = 0

    for page in range(1, max_pages + 1):
        listing_url = f"https://www.sharesansar.com/category/latest?page={page}"
        try:
            resp = requests.get(listing_url, headers=HEADERS, timeout=20)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, "lxml")

            page_links = _extract_news_links(
                soup=soup,
                base_url=listing_url,
                allowed_domains=("sharesansar.com",),
                selectors=(
                    "a.news-title[href]",
                    "h4 a[href]",
                    "h3 a[href]",
                    "article a[href]",
                    ".media a[href]",
                ),
                path_keywords=("news", "detail", "article", "announcement"),
            )

            if not page_links:
                empty_streak += 1
                print(f"Sharesansar page {page}: 0 links")
                if empty_streak >= 3:
                    print("Stopping Sharesansar pagination after 3 empty pages.")
                    break
            else:
                empty_streak = 0
                urls.update(page_links)
                print(f"Sharesansar page {page}: +{len(page_links)} links (total: {len(urls)})")

            time.sleep(SCRAPE_DELAY_SEC)
        except Exception as e:
            print(f"Sharesansar error on page {page}: {e}")
            time.sleep(SCRAPE_DELAY_SEC * 2)

    return sorted(urls)


def _first_match(soup: BeautifulSoup, selectors: tuple[str, ...]):
    for sel in selectors:
        el = soup.select_one(sel)
        if el:
            return el
    return None


def fetch_and_parse_article(url: str, source: str = "sharesansar") -> dict | None:
    """Fetch one article and parse it into the ChromaDB document schema."""
    cache_key = hashlib.md5(url.encode()).hexdigest()
    cache_path = os.path.join(NEWS_CACHE_DIR, f"{cache_key}.html")

    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            html = f.read()
    else:
        try:
            resp = requests.get(url, headers=HEADERS, timeout=20)
            resp.raise_for_status()
            html = resp.text
            with open(cache_path, "w", encoding="utf-8") as f:
                f.write(html)
            time.sleep(SCRAPE_DELAY_SEC)
        except Exception as e:
            print(f"Fetch failed: {url} - {e}")
            return None

    soup = BeautifulSoup(html, "lxml")

    title_el = _first_match(
        soup,
        (
            "h1.news-heading",
            "h1.entry-title",
            "h1.post-title",
            "article h1",
            "section.main-content h1",
            "h1",
        ),
    )
    content_el = _first_match(
        soup,
        (
            "div.news-content",
            "div.entry-content",
            "div.post-content",
            "div.text-justify",
            "section.main-content",
            "article .content",
            "article",
        ),
    )
    date_el = _first_match(
        soup,
        (
            "span.news-date",
            "time",
            ".entry-date",
            "section.main-content time",
            "meta[property='article:published_time']",
            "meta[name='pubdate']",
        ),
    )

    if not title_el or not content_el:
        return None

    title = title_el.get_text(strip=True)

    # Prefer paragraph text to avoid menu/nav text in wrapper sections.
    paragraphs = [p.get_text(" ", strip=True) for p in content_el.select("p") if p.get_text(strip=True)]
    content = " ".join(paragraphs) if paragraphs else content_el.get_text(" ", strip=True)

    # Remove common site chrome words if they were included by wrapper selectors.
    for noise in ("Login", "Register", "Contact Us", "Write For Us", "Home News", "Latest Share Price"):
        content = content.replace(noise, " ")
    content = " ".join(content.split())

    if len(content) < 80:
        return None

    published_at = ""
    raw_date = ""
    if date_el:
        raw_date = date_el.get("content", "").strip() if date_el.name == "meta" else date_el.get_text(strip=True)

    if raw_date:
        if "T" in raw_date:
            raw_date = raw_date.split("T")[0]
        for fmt in ("%Y-%m-%d", "%B %d, %Y", "%d %b %Y", "%d %B %Y", "%Y/%m/%d", "%a, %b %d, %Y"):
            try:
                published_at = datetime.strptime(raw_date, fmt).strftime("%Y-%m-%d")
                break
            except ValueError:
                continue

    try:
        language = detect_language(title + " " + content[:200])
    except Exception:
        language = "unknown"

    from nepse_analyst.database import get_connection

    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT symbol, name, sector FROM companies")
    companies = cursor.fetchall()
    conn.close()

    detected_symbol = None
    detected_sector = None
    text_upper = (title + " " + content[:500]).upper()
    for symbol, name, sector in companies:
        if symbol in text_upper or name.upper() in text_upper:
            detected_symbol = symbol
            detected_sector = sector
            break

    title_lower = title.lower()
    if any(k in title_lower for k in ["dividend", "लाभांश"]):
        article_type = "dividend"
    elif any(k in title_lower for k in ["ipo", "public issue", "सार्वजनिक निष्काशन"]):
        article_type = "ipo"
    elif any(k in title_lower for k in ["agm", "annual general", "साधारण सभा"]):
        article_type = "agm"
    elif any(k in title_lower for k in ["earning", "profit", "नाफा", "quarter"]):
        article_type = "earnings"
    elif any(k in title_lower for k in ["sebon", "regulation", "नियामक", "policy"]):
        article_type = "regulatory"
    else:
        article_type = "market"

    date_slug = published_at.replace("-", "") if published_at else "undated"
    doc_id = f"{source}_{date_slug}_{cache_key[:8]}"

    return {
        "id": doc_id,
        "source": source,
        "title": title,
        "content": content,
        "symbol": detected_symbol,
        "sector": detected_sector,
        "language": language,
        "published_at": published_at,
        "article_type": article_type,
        "url": url,
    }

In [6]:
def get_article_urls_nepse_alpha(max_pages: int = 20) -> list[str]:
    """Collect article URLs from NepseAlpha (supports both known domains)."""
    urls = set()
    empty_streak = 0

    listing_patterns = (
        "https://www.nepsealpha.com/news/page/{page}/",
        "https://www.nepsealpha.com/category/news/page/{page}/",
        "https://www.nepse-alpha.com/news/page/{page}/",
    )

    for page in range(1, max_pages + 1):
        listing_url = None
        resp = None

        # Try alternate listing URL patterns for this page.
        for pattern in listing_patterns:
            candidate = pattern.format(page=page)
            try:
                r = requests.get(candidate, headers=HEADERS, timeout=20)
                if r.status_code == 200 and "<html" in r.text.lower():
                    listing_url = candidate
                    resp = r
                    break
            except Exception:
                continue

        if not resp or not listing_url:
            empty_streak += 1
            print(f"NepseAlpha page {page}: no reachable listing URL")
            if empty_streak >= 3:
                print("Stopping NepseAlpha pagination after 3 empty pages.")
                break
            continue

        soup = BeautifulSoup(resp.text, "lxml")
        page_links = _extract_news_links(
            soup=soup,
            base_url=listing_url,
            allowed_domains=("nepsealpha.com", "nepse-alpha.com"),
            selectors=(
                "article a[href]",
                "h2 a[href]",
                "h3 a[href]",
                ".post a[href]",
                ".news a[href]",
            ),
            path_keywords=("news", "announcement", "market", "article"),
        )

        if not page_links:
            empty_streak += 1
            print(f"NepseAlpha page {page}: 0 links")
            if empty_streak >= 3:
                print("Stopping NepseAlpha pagination after 3 empty pages.")
                break
        else:
            empty_streak = 0
            urls.update(page_links)
            print(f"NepseAlpha page {page}: +{len(page_links)} links (total: {len(urls)})")

        time.sleep(SCRAPE_DELAY_SEC)

    return sorted(urls)

In [7]:
from tqdm import tqdm

all_docs = []

# Sharesansar
print("Collecting Sharesansar article URLs...")
ss_urls = get_article_urls_sharesansar(max_pages=100)
print(f"Found {len(ss_urls)} Sharesansar URLs")

print("Fetching and parsing Sharesansar articles...")
for url in tqdm(ss_urls[:MAX_ARTICLES]):
    doc = fetch_and_parse_article(url, source="sharesansar")
    if doc:
        all_docs.append(doc)

# NepseAlpha
print("Collecting NepseAlpha article URLs...")
na_urls = get_article_urls_nepse_alpha(max_pages=30)
print(f"Found {len(na_urls)} NepseAlpha URLs")

for url in tqdm(na_urls):
    doc = fetch_and_parse_article(url, source="nepse_alpha")
    if doc:
        all_docs.append(doc)

print(f"\nTotal documents collected: {len(all_docs)}")
print(f"Languages: {set(d['language'] for d in all_docs)}")
print(f"Article types: {set(d['article_type'] for d in all_docs)}")

Sharesansar page 1: +16 links (total: 16)
Sharesansar page 2: +16 links (total: 16)
Sharesansar page 3: +16 links (total: 16)
Sharesansar page 4: +16 links (total: 16)
Sharesansar page 5: +16 links (total: 16)
Sharesansar page 6: +16 links (total: 16)
Sharesansar page 7: +16 links (total: 16)
Sharesansar page 8: +16 links (total: 16)
Sharesansar page 9: +16 links (total: 16)
Sharesansar page 10: +16 links (total: 16)
Sharesansar page 11: +16 links (total: 16)
Sharesansar page 12: +16 links (total: 16)
Sharesansar page 13: +16 links (total: 16)
Sharesansar page 14: +16 links (total: 16)
Sharesansar page 15: +16 links (total: 16)
Sharesansar page 16: +16 links (total: 16)
Sharesansar page 17: +16 links (total: 16)
Sharesansar page 18: +16 links (total: 16)
Sharesansar page 19: +16 links (total: 16)
Sharesansar page 20: +16 links (total: 16)
Sharesansar page 21: +16 links (total: 16)
Sharesansar page 22: +16 links (total: 16)
Sharesansar page 23: +16 links (total: 16)
Sharesansar page 24:

100%|██████████| 16/16 [00:51<00:00,  3.24s/it]


NepseAlpha page 1: no reachable listing URL
NepseAlpha page 2: no reachable listing URL
NepseAlpha page 3: no reachable listing URL
Stopping NepseAlpha pagination after 3 empty pages.
Found 0 NepseAlpha URLs


0it [00:00, ?it/s]


Total documents collected: 16
Languages: {'en'}
Article types: {'regulatory', 'ipo', 'market'}


In [8]:
def deduplicate(docs: list[dict]) -> list[dict]:
    seen_titles = set()
    unique = []
    for doc in docs:
        key = doc["title"].lower().strip()
        if key not in seen_titles:
            seen_titles.add(key)
            unique.append(doc)
    return unique


all_docs = deduplicate(all_docs)
print(f"After deduplication: {len(all_docs)} documents")

# Save the raw document list as a local JSON backup before embedding
docs_backup_path = os.path.join(BASE_DIR, "data", "processed", "all_docs.json")
with open(docs_backup_path, "w", encoding="utf-8") as f:
    json.dump(all_docs, f, ensure_ascii=False, indent=2)
print(f"Raw documents saved to {docs_backup_path}")

After deduplication: 16 documents
Raw documents saved to /home/meutsabdahal/Desktop/nepse-analyst/data/processed/all_docs.json


In [9]:
import os
import numpy as np
from sklearn.feature_extraction.text import HashingVectorizer

# Prevent implicit use of stale Hugging Face auth tokens.
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
for token_var in ("HF_TOKEN", "HUGGINGFACEHUB_API_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
    os.environ.pop(token_var, None)

# Compatibility shim: chromadb 0.5.x expects np.float_ (removed in NumPy 2.x).
if not hasattr(np, "float_"):
    np.float_ = np.float64

import chromadb
from nepse_analyst.embeddings import encode

# Local fallback embedder if HF model download/auth fails.
_fallback_vectorizer = HashingVectorizer(n_features=384, alternate_sign=False, norm="l2")
def _fallback_encode(texts: list[str]):
    return _fallback_vectorizer.transform(texts).toarray().astype("float32")

# Initialise ChromaDB with local persistence
chroma_client = chromadb.PersistentClient(path=VECTOR_STORE_DIR)

# Delete existing collection if rebuilding
try:
    chroma_client.delete_collection("nepse_news")
    print("Deleted existing collection.")
except:
    pass

collection = chroma_client.create_collection(
    name="nepse_news",
    metadata={"hnsw:space": "cosine"}  # use cosine similarity
)

# Process in batches - ChromaDB handles large inserts better in chunks
BATCH_SIZE = 200
use_fallback_embeddings = False

print(f"Embedding and indexing {len(all_docs)} documents...")
for i in tqdm(range(0, len(all_docs), BATCH_SIZE)):
    batch = all_docs[i:i + BATCH_SIZE]

    # Embed: use title + first 300 chars of content for the embedding
    texts_to_embed = [f"{doc['title']}. {doc['content'][:300]}" for doc in batch]

    if use_fallback_embeddings:
        embeddings = _fallback_encode(texts_to_embed)
    else:
        try:
            embeddings = encode(texts_to_embed, batch_size=64)
        except Exception as e:
            print(f"Primary embedder failed; switching to hashing fallback. Reason: {e}")
            use_fallback_embeddings = True
            embeddings = _fallback_encode(texts_to_embed)

    collection.add(
        ids=[doc["id"] for doc in batch],
        embeddings=[emb.tolist() for emb in embeddings],
        documents=[doc["content"] for doc in batch],
        metadatas=[
            {
                "source": doc["source"],
                "title": doc["title"],
                "symbol": doc["symbol"] or "",
                "sector": doc["sector"] or "",
                "language": doc["language"],
                "published_at": doc["published_at"],
                "article_type": doc["article_type"],
                "url": doc["url"],
            }
            for doc in batch
        ],
    )

print(f"Indexed {collection.count()} documents into ChromaDB.")

/home/meutsabdahal/Desktop/nepse-analyst/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Deleted existing collection.
Embedding and indexing 16 documents...


  0%|          | 0/1 [00:00<?, ?it/s]

Loading embedding model: paraphrase-multilingual-MiniLM-L12-v2 (first call only)...
Embedding model loaded.


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given
100%|██████████| 1/1 [00:06<00:00,  6.76s/it]

Indexed 16 documents into ChromaDB.


In [10]:
import shutil

# ChromaDB PersistentClient writes to disk automatically, but confirm
dir_size_mb = sum(
    os.path.getsize(os.path.join(VECTOR_STORE_DIR, f))
    for f in os.listdir(VECTOR_STORE_DIR)
    if os.path.isfile(os.path.join(VECTOR_STORE_DIR, f))
) / 1024 / 1024
print(f"ChromaDB directory size: {dir_size_mb:.1f} MB")

# Keep a local backup copy instead of a Colab Drive path
backup_dest = os.path.join(BASE_DIR, 'data', 'processed', 'vector_store_backup')
if os.path.exists(backup_dest):
    shutil.rmtree(backup_dest)
shutil.copytree(VECTOR_STORE_DIR, backup_dest)
print(f"ChromaDB backup saved to {backup_dest}")

ChromaDB directory size: 1.0 MB
ChromaDB backup saved to /home/meutsabdahal/Desktop/nepse-analyst/data/processed/vector_store_backup
